In [0]:
dbutils.library.restartPython()

In [0]:
# ==========================================
# DATABRICKS WORKSPACE CONFIGURATION
# ==========================================

# Get current notebook context
context = (
    dbutils.notebook
    .entry_point
    .getDbutils()
    .notebook()
    .getContext()
)

# Retrieve workspace URL
try:
    workspace_host = spark.conf.get(
        "spark.databricks.workspaceUrl"
    )
except Exception:
    workspace_host = context.browserHostName().get()

workspace_url = (
    workspace_host
    if workspace_host.startswith("https://")
    else f"https://{workspace_host}"
)

# Retrieve notebook authentication token
databricks_token = context.apiToken().get()

if not workspace_url or not databricks_token:
    raise RuntimeError(
        "Databricks workspace URL or authentication token is unavailable."
    )

print("Workspace URL:", workspace_url)
print("Authentication token: Retrieved successfully")

In [0]:
from LLM.TaskflowCodeGenerator import TaskflowCodeGenerator

OUTPUT_DIR = "/Volumes/idmc_poc/bronze/idmc_files"

generator = TaskflowCodeGenerator(
    workspace_url=workspace_url,
    databricks_token=databricks_token,
    model="databricks-gpt-oss-120b",
    max_tokens=12000,
    temperature=0.1
)

print("TaskflowCodeGenerator initialized successfully.")

In [0]:
XML_PATH = "/Volumes/idmc_poc/bronze/idmc_files/wf_s_m_TIM_STG_UNQ_WRKSTN_NOMTCH_SYNC.TASKFLOW_SANITIZE.xml"

print("XML_PATH:", XML_PATH)

In [0]:
# STEP 1 — Read and parse Taskflow XML

from parser.xmlreader import XMLReader
from parser.Taskflow_parser import TaskflowParser

# Confirm XML path
if "XML_PATH" not in globals():
    raise NameError("XML_PATH is not defined. Set your Taskflow XML path.")

# Initialize XMLReader with the required path
xml_reader = XMLReader(XML_PATH)

# Read XML
root = xml_reader.read()

print("XML loaded successfully.")
print("Root element:", root.tag)

# Parse Taskflow
taskflow_parser = TaskflowParser()
parsed_taskflow = taskflow_parser.parse(root)

print("\nTaskflow parsing completed.")
print("Output type:", type(parsed_taskflow))

if isinstance(parsed_taskflow, dict):
    print("Output keys:", list(parsed_taskflow.keys()))
    print("Summary:", parsed_taskflow.get("summary"))

In [0]:
import inspect
from parser.graph_builder import GraphBuilder

print("GraphBuilder class:", GraphBuilder)

print("\nAvailable methods:")
for name, method in inspect.getmembers(
    GraphBuilder,
    predicate=inspect.isfunction
):
    if not name.startswith("__"):
        print(f"{name}{inspect.signature(method)}")

print("\nParsed Taskflow structure:")
print("Top-level keys:", list(parsed_taskflow.keys()))
print("Taskflow keys:", list(parsed_taskflow["taskflow"].keys()))

In [0]:
# STEP 3 — Inspect Taskflow services, control flow and links

import json

taskflow_data = parsed_taskflow["taskflow"]

print("TASKFLOW SERVICES")
print("Count:", len(taskflow_data.get("services", [])))
print(json.dumps(
    taskflow_data.get("services", [])[:2],
    indent=2,
    ensure_ascii=False,
    default=str
))

print("\nCONTROL FLOW NODES")
print("Count:", len(taskflow_data.get("control_flow_nodes", [])))
print(json.dumps(
    taskflow_data.get("control_flow_nodes", [])[:2],
    indent=2,
    ensure_ascii=False,
    default=str
))

print("\nLINKS")
print("Count:", len(taskflow_data.get("links", [])))
print(json.dumps(
    taskflow_data.get("links", [])[:3],
    indent=2,
    ensure_ascii=False,
    default=str
))

print("\nOPERATIONS")
print("Count:", len(taskflow_data.get("operations", [])))
print(json.dumps(
    taskflow_data.get("operations", [])[:3],
    indent=2,
    ensure_ascii=False,
    default=str
))

In [0]:
# STEP 4 — Inspect Taskflow dependency and flow structures

import json

taskflow_data = parsed_taskflow["taskflow"]

for key in ["dependencies", "flow", "attributes", "generator"]:
    value = taskflow_data.get(key)

    print(f"\n{'=' * 60}")
    print(f"{key.upper()}")
    print("Type:", type(value).__name__)

    if isinstance(value, list):
        print("Count:", len(value))
        print(json.dumps(value[:5], indent=2, ensure_ascii=False, default=str))

    elif isinstance(value, dict):
        print("Keys:", list(value.keys()))
        print(json.dumps(value, indent=2, ensure_ascii=False, default=str)[:12000])

    else:
        print(value)

In [0]:

# STEP 5 — Corrected Taskflow graph and LLM input

import json
import sys

PROJECT_ROOT = (
    "/Workspace/Users/lokeswaraseetaramasastry.k@tcs.com/"
    "IDMC_Databricks_LLM"
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

taskflow_data = parsed_taskflow["taskflow"]


def walk_flow(element, parent_id=None, nodes=None, edges=None):
    if nodes is None:
        nodes = []
    if edges is None:
        edges = []

    if not isinstance(element, dict):
        return nodes, edges

    tag = element.get("tag")
    attrs = element.get("attributes") or {}
    node_id = attrs.get("id")

    if (
        tag
        and tag not in {"flow", "title", "operation", "expression", "link"}
        and node_id
    ):
        title = next(
            (
                child.get("text")
                for child in element.get("children", [])
                if child.get("tag") == "title"
            ),
            None
        )

        nodes.append({
            "id": node_id,
            "element_type": tag,
            "title": title or tag,
            "attributes": attrs
        })

        if parent_id:
            edges.append({
                "source": parent_id,
                "target": node_id,
                "edge_type": "nested_containment",
                "evidence": "nested_flow_xml"
            })

        parent_id = node_id

    for child in element.get("children", []):
        walk_flow(child, parent_id, nodes, edges)

    return nodes, edges


flow_nodes, containment_edges = walk_flow(
    taskflow_data.get("flow")
)

# IMPORTANT:
# Do not add services to graph nodes.
# The Taskflow builder already reads services from parsed_taskflow.

taskflow_graph = {
    "nodes": flow_nodes,
    "containment_edges": containment_edges,
    "dependency_edges": [],
    "condition_dependency_edges": [],
    "unresolved_links": taskflow_data.get("links", []),
    "link_inventory": taskflow_data.get("links", []),
    "flows": taskflow_data.get("flow"),
    "summary": {
        "flow_node_count": len(flow_nodes),
        "containment_edge_count": len(containment_edges),
        "service_count_in_graph": 0,
        "dependency_edges_confirmed": 0,
        "link_count": len(taskflow_data.get("links", []))
    }
}

print("TASKFLOW GRAPH")
print(json.dumps(taskflow_graph["summary"], indent=2))

from LLM.Taskflow_Input_builder import TaskflowLLMInputBuilder

builder = TaskflowLLMInputBuilder()

taskflow_input = builder.build(
    parsed_taskflow,
    taskflow_graph
)

print("\nTASKFLOW INPUT BUILT SUCCESSFULLY")
print("Input keys:", list(taskflow_input.keys()))
print(
    "Parser evidence keys:",
    list(taskflow_input.get("parser_evidence", {}).keys())
)


In [0]:

# STEP 6 — Generate Taskflow PySpark

from pathlib import Path
import json
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

result = generator.generate_taskflow_code(
    taskflow_input=taskflow_input,
    output_dir=OUTPUT_DIR,
    max_retries=1
)

print("GENERATION COMPLETED")
print(json.dumps(result, indent=2, default=str))

print("\nFILES CREATED IN OUTPUT DIRECTORY")
display(dbutils.fs.ls(OUTPUT_DIR))


In [0]:

# STEP 5 — Fix Taskflow service metadata and rebuild LLM input

from LLM.Taskflow_Input_builder import TaskflowLLMInputBuilder

taskflow_data = parsed_taskflow["taskflow"]

# Preserve the graph already constructed in Step 5.
# Do not pass this Taskflow into the mapping GraphBuilder.

taskflow_graph["nodes"] = flow_nodes
taskflow_graph["link_inventory"] = taskflow_data.get("links", [])
taskflow_graph["unresolved_links"] = taskflow_data.get("links", [])

builder = TaskflowLLMInputBuilder()

taskflow_input = builder.build(
    parsed_taskflow,
    taskflow_graph
)

# Replace incomplete graph-derived service records
# with the complete parser service metadata.
parser_services = taskflow_data.get("services", [])

taskflow_input["parser_evidence"]["services"] = parser_services
taskflow_input["input_integrity"]["service_count"] = len(parser_services)

# Verify classification BEFORE calling the LLM.
from LLM.TaskflowCodeGenerator import TaskflowCodeGenerator

compact = TaskflowCodeGenerator.compact_input(taskflow_input)

print("SERVICE COUNT:", compact["counts"]["service_count"])
print("DATA TASKS:", compact["counts"]["data_task_count"])
print("COMMAND TASKS:", compact["counts"]["command_task_count"])

for service in compact["services"]:
    print(
        service.get("title"),
        "|",
        service.get("service_name"),
        "|",
        TaskflowCodeGenerator._service_type(service)
    )

assert compact["counts"]["service_count"] == 15
assert compact["counts"]["data_task_count"] == 5
assert compact["counts"]["command_task_count"] == 10

print("SERVICE CLASSIFICATION: PASS")


In [0]:
# STEP 6 — Regenerate using corrected service metadata

result = generator.generate_taskflow_code(
    taskflow_input=taskflow_input,
    output_dir=OUTPUT_DIR,
    max_retries=1
)

print("GENERATION COMPLETED")
print(json.dumps(result, indent=2, default=str))

In [0]:
generated_path = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_wf_s_m_tim_stg_unq_wrkstn_nomtch_sync.py"
)

with open(generated_path, "r", encoding="utf-8") as f:
    generated_code = f.read()

failed_functions = [
    "execute_s_m_tim_stg_unq_wrkstn_nomtch_sync",
    "execute_s_m_tdr_stg_src_tgt_cnt_gen",
    "execute_s_m_tdr_upd_config_evnt_batch_log_dtls"
]

for name in failed_functions:
    print("\n", "=" * 80)
    print("FUNCTION:", name)
    print("PRESENT:", f"def {name}(" in generated_code)

In [0]:
import ast

generated_path = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_wf_s_m_tim_stg_unq_wrkstn_nomtch_sync.py"
)

with open(generated_path, "r", encoding="utf-8") as f:
    generated_code = f.read()

tree = ast.parse(generated_code)

failed_functions = [
    "execute_s_m_tim_stg_unq_wrkstn_nomtch_sync",
    "execute_s_m_tdr_stg_src_tgt_cnt_gen",
    "execute_s_m_tdr_upd_config_evnt_batch_log_dtls"
]

for function_name in failed_functions:
    print("\n" + "=" * 100)
    print("FUNCTION:", function_name)

    function_node = next(
        (
            node for node in tree.body
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == function_name
        ),
        None
    )

    if function_node is None:
        print("FUNCTION NOT FOUND")
        continue

    lines = generated_code.splitlines()
    start = function_node.lineno - 1
    end = min(function_node.end_lineno, start + 45)

    print("\n".join(lines[start:end]))

In [0]:
print("FUNCTION SOURCE:")
print(function_source)

print("\nFUNCTION NAME:", fn.name)
print("\nSERVICE:", item["title"])
print("\nSERVICE ID:", item["service_id"])

In [0]:
print("=== RETRY STATE ===")

for name in [
    "taskflow_input",
    "generator",
    "function_source",
    "item",
    "evidence",
    "generated_code",
    "report",
    "metadata"
]:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'NOT AVAILABLE'}")

print("\n=== SECOND SERVICE ===")
print("Function source available:", "function_source" in globals())

if "function_source" in globals():
    import ast
    tree = ast.parse(function_source)

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            print("Function:", node.name)
            print("Syntax: VALID")

In [0]:
import os
import json

OUTPUT_DIR = "/Volumes/idmc_poc/bronze/idmc_files"

REPORT_PATH = os.path.join(
    OUTPUT_DIR, "taskflow_generation_report.json"
)

METADATA_PATH = os.path.join(
    OUTPUT_DIR, "taskflow_generation_metadata.json"
)

CODE_PATH = os.path.join(
    OUTPUT_DIR,
    "generated_wf_s_m_tim_stg_unq_wrkstn_nomtch_sync.py"
)

for path in [CODE_PATH, REPORT_PATH, METADATA_PATH]:
    print(
        os.path.basename(path),
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

if os.path.exists(REPORT_PATH):
    with open(REPORT_PATH, "r") as f:
        report = json.load(f)

if os.path.exists(METADATA_PATH):
    with open(METADATA_PATH, "r") as f:
        metadata = json.load(f)

print("\nReport loaded:", "report" in globals())
print("Metadata loaded:", "metadata" in globals())

In [0]:
import inspect
import json

TARGET_SERVICE_ID = "ms07zo8r"
TARGET_SERVICE_NAME = "s_m_TDR_UPD_CONFIG_EVNT_BATCH_LOG_DTLS"

print("=== GENERATOR METHODS ===")

for method_name in [
    "_service_evidence",
    "_build_service_prompt",
    "_call_llm",
    "_extract_code"
]:
    method = getattr(generator, method_name, None)

    if method:
        print(method_name, inspect.signature(method))
    else:
        print(method_name, "NOT FOUND")

print("\n=== TARGET SERVICE ===")

services = taskflow_input["parser_evidence"]["services"]

target_service = next(
    (
        service for service in services
        if service.get("id") == TARGET_SERVICE_ID
        or service.get("service_id") == TARGET_SERVICE_ID
    ),
    None
)

if target_service is None:
    target_service = next(
        (
            service for service in services
            if TARGET_SERVICE_NAME in json.dumps(service)
        ),
        None
    )

print(json.dumps(target_service, indent=2, default=str))

In [0]:
import ast
import os

CODE_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_wf_s_m_tim_stg_unq_wrkstn_nomtch_sync.py"
)

with open(CODE_PATH, "r", encoding="utf-8") as f:
    code = f.read()

tree = ast.parse(code)

functions = [
    node for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
]

pyspark_patterns = {
    "Spark imports": ["from pyspark", "import pyspark"],
    "Spark reads": ["spark.table(", "spark.read.", "spark.sql("],
    "DataFrame transformations": [
        ".select(", ".filter(", ".where(", ".join(",
        ".withColumn(", ".groupBy(", ".agg(",
        ".dropDuplicates("
    ],
    "Spark writes": [
        ".write.", ".writeTo(", "saveAsTable("
    ],
    "Placeholders": [
        "GENERATION_FAILED", "NotImplementedError",
        "TODO", "BLOCKED", "PARTIAL"
    ]
}

print("=== GENERATED FILE ANALYSIS ===")
print("File:", os.path.basename(CODE_PATH))
print("Total lines:", len(code.splitlines()))
print("Functions:", len(functions))
print("Python syntax:", "PASS")

for category, patterns in pyspark_patterns.items():
    matches = [p for p in patterns if p in code]
    print(f"{category}: {matches if matches else 'NOT FOUND'}")

print("\n=== FUNCTION-LEVEL PYSPARK ANALYSIS ===")

for fn in functions:
    source = ast.get_source_segment(code, fn) or ""

    has_read = any(
        p in source
        for p in pyspark_patterns["Spark reads"]
    )

    has_transform = any(
        p in source
        for p in pyspark_patterns["DataFrame transformations"]
    )

    has_write = any(
        p in source
        for p in pyspark_patterns["Spark writes"]
    )

    has_placeholder = any(
        p in source
        for p in pyspark_patterns["Placeholders"]
    )

    print(f"\nFunction: {fn.name}")
    print("  Spark Read:", has_read)
    print("  DataFrame Transform:", has_transform)
    print("  Spark Write:", has_write)
    print("  Placeholder/Partial marker:", has_placeholder)

In [0]:
import ast

tree = ast.parse(code)

data_task_functions = [
    "execute_s_m_tdr_load_prcs_btch_dtl_log",
    "execute_s_m_tim_stg_unq_wrkstn_nomtch_sync",
    "execute_s_m_tdr_stg_src_tgt_cnt_gen",
    "execute_s_m_tdr_load_prcs_batch_file_dtls",
    "execute_s_m_tdr_upd_config_evnt_batch_log_dtls"
]

for node in tree.body:
    if isinstance(node, ast.FunctionDef) and node.name in data_task_functions:
        print("\n" + "=" * 90)
        print("FUNCTION:", node.name)
        print("=" * 90)
        print(ast.get_source_segment(code, node))

In [0]:
import inspect
import json

print("=== MIGRATION REPORT FUNCTION ===")

print(inspect.getsource(migration_report))

print("\n=== REPORT METRICS ===")

print(json.dumps(
    report.get("metrics", {}),
    indent=2,
    default=str
))

print("\n=== PER SERVICE RESULTS ===")

print(json.dumps(
    report.get("per_service_results", []),
    indent=2,
    default=str
))

In [0]:
import ast

with open(CODE_PATH, "r", encoding="utf-8") as f:
    source = f.read()

tree = ast.parse(source)

for node in tree.body:
    if isinstance(node, ast.FunctionDef) and node.name == "migration_report":
        print("=== MIGRATION REPORT FUNCTION ===")
        print(ast.get_source_segment(source, node))
        break
else:
    print("migration_report function not found")

In [0]:
import json

with open(REPORT_PATH, "r", encoding="utf-8") as f:
    report_data = json.load(f)

print("=== METRICS ===")
print(json.dumps(report_data.get("metrics", {}), indent=2))

print("\n=== VALIDATION ===")
print(json.dumps(report_data.get("validation", {}), indent=2))

print("\n=== PER SERVICE RESULTS ===")
print(json.dumps(report_data.get("per_service_results", []), indent=2))

In [0]:
print("=== REPORT JSON STRUCTURE ===")

print("Top-level keys:", list(report_data.keys()))

for key, value in report_data.items():
    print(f"\n{key}: {type(value).__name__}")

    if isinstance(value, dict):
        print("  Keys:", list(value.keys()))

    elif isinstance(value, list):
        print("  Item count:", len(value))
        if value:
            print("  First item:", value[0])

In [0]:
import json

metrics = report_data.get("generation_metrics", {})

print("========== TASKFLOW ENGINE PERFORMANCE ==========")

print("\n--- XML PARSING ---")
print(json.dumps(report_data.get("parsed_counts", {}), indent=2))

print("\n--- GENERATION METRICS ---")

for key, value in metrics.items():
    if key not in ["checks", "per_service_generation", "operation_coverage"]:
        print(f"{key}: {value}")

print("\n--- OPERATION COVERAGE ---")
print(json.dumps(metrics.get("operation_coverage", {}), indent=2))

print("\n--- VALIDATION CHECKS ---")
print(json.dumps(metrics.get("checks", {}), indent=2))

print("\n--- SERVICE GENERATION RESULTS ---")

for service in report_data.get("per_service_generation", []):
    print(
        service.get("service"),
        "|",
        service.get("service_type"),
        "|",
        service.get("status")
    )

print("\n--- GENERATION ERRORS ---")
print(json.dumps(report_data.get("generation_errors", []), indent=2))

print("\n--- ASSUMPTIONS ---")
print(json.dumps(report_data.get("assumptions", []), indent=2))

print("\n--- UNSUPPORTED ITEMS ---")
print(json.dumps(report_data.get("unsupported_items", []), indent=2))

In [0]:

import ast
import os
import json

# ============================================================
# CONFIGURATION
# ============================================================

CODE_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_wf_s_m_tim_stg_unq_wrkstn_nomtch_sync.py"
)

CATALOG = "idmc_poc"

DATA_TASKS = [
    "execute_s_m_tdr_load_prcs_btch_dtl_log",
    "execute_s_m_tim_stg_unq_wrkstn_nomtch_sync",
    "execute_s_m_tdr_stg_src_tgt_cnt_gen",
    "execute_s_m_tdr_load_prcs_batch_file_dtls",
    "execute_s_m_tdr_upd_config_evnt_batch_log_dtls"
]

SCHEMAS_TO_INSPECT = [
    "bronze",
    "silver",
    "curated",
    "reference",
    "sales"
]

# ============================================================
# 1. LOAD GENERATED CODE
# ============================================================

if not os.path.exists(CODE_PATH):
    raise FileNotFoundError(CODE_PATH)

with open(CODE_PATH, "r", encoding="utf-8") as file:
    generated_code = file.read()

tree = ast.parse(generated_code)

function_map = {
    node.name: node
    for node in tree.body
    if isinstance(node, ast.FunctionDef)
}

print("=" * 75)
print("TASKFLOW DATA-TASK EXECUTION READINESS")
print("=" * 75)

# ============================================================
# 2. ANALYZE EACH GENERATED DATA-TASK FUNCTION
# ============================================================

read_methods = {
    "table", "load", "json", "csv", "parquet",
    "format", "sql"
}

transform_methods = {
    "select", "filter", "where", "join",
    "withColumn", "withColumnRenamed",
    "drop", "dropDuplicates", "distinct",
    "groupBy", "agg", "union", "unionByName",
    "orderBy", "sort", "fillna", "na"
}

write_methods = {
    "save", "saveAsTable", "insertInto",
    "parquet", "json", "csv", "format"
}

readiness_results = []

for function_name in DATA_TASKS:

    node = function_map.get(function_name)

    if node is None:
        result = {
            "function": function_name,
            "status": "MISSING_FUNCTION",
            "reads": [],
            "transformations": [],
            "writes": [],
            "issues": ["Function was not generated"]
        }

        readiness_results.append(result)
        continue

    calls = [
        child for child in ast.walk(node)
        if isinstance(child, ast.Call)
    ]

    called_methods = set()
    raised_errors = []
    imports = []

    for call in calls:
        if isinstance(call.func, ast.Attribute):
            called_methods.add(call.func.attr)

        elif isinstance(call.func, ast.Name):
            if call.func.id in {
                "NotImplementedError", "ValueError"
            }:
                raised_errors.append(call.func.id)

    for child in ast.walk(node):
        if isinstance(child, ast.ImportFrom):
            if child.module and child.module.startswith("pyspark"):
                imports.append(child.module)

        if isinstance(child, ast.Raise):
            if isinstance(child.exc, ast.Call):
                if isinstance(child.exc.func, ast.Name):
                    raised_errors.append(child.exc.func.id)

    reads = sorted(called_methods & read_methods)
    transformations = sorted(
        called_methods & transform_methods
    )
    writes = sorted(called_methods & write_methods)

    function_source = ast.get_source_segment(
        generated_code, node
    ) or ""

    if "GENERATION_FAILED" in function_source:
        status = "GENERATION_FAILED"

    elif "NotImplementedError" in raised_errors:
        status = "BLOCKED_STUB"

    elif reads and transformations and writes:
        status = "PYSPARK_CANDIDATE"

    elif reads or transformations or writes:
        status = "PARTIAL_PYSPARK"

    else:
        status = "NO_VERIFIED_PYSPARK_LOGIC"

    issues = []

    if not reads:
        issues.append("No verified source-read operation")

    if not transformations:
        issues.append("No verified DataFrame transformation")

    if not writes:
        issues.append("No verified target-write operation")

    if raised_errors:
        issues.append(
            "Explicit exceptions: " + ", ".join(set(raised_errors))
        )

    result = {
        "function": function_name,
        "status": status,
        "reads": reads,
        "transformations": transformations,
        "writes": writes,
        "pyspark_imports": imports,
        "issues": issues
    }

    readiness_results.append(result)

    print("\nFunction:", function_name)
    print("Status:", status)
    print("Reads:", reads)
    print("Transformations:", transformations)
    print("Writes:", writes)
    print("Issues:", issues)

# ============================================================
# 3. INVENTORY EXISTING DATABRICKS TABLES
# ============================================================

print("\n" + "=" * 75)
print("AVAILABLE TABLE INVENTORY")
print("=" * 75)

table_inventory = []

for schema_name in SCHEMAS_TO_INSPECT:

    try:
        query = f"SHOW TABLES IN `{CATALOG}`.`{schema_name}`"

        tables_df = spark.sql(query)

        tables = [
            row.tableName
            for row in tables_df.collect()
        ]

        for table_name in tables:
            table_inventory.append({
                "catalog": CATALOG,
                "schema": schema_name,
                "table": table_name,
                "full_name": (
                    f"{CATALOG}.{schema_name}.{table_name}"
                )
            })

        print(f"\n{schema_name}: {len(tables)} tables")

        for table_name in tables:
            print("  -", table_name)

    except Exception as exc:
        print(
            f"\n{schema_name}: UNAVAILABLE - "
            f"{str(exc)[:250]}"
        )

# ============================================================
# 4. SUMMARIZE EXECUTION READINESS
# ============================================================

print("\n" + "=" * 75)
print("FINAL READINESS SUMMARY")
print("=" * 75)

for status in [
    "PYSPARK_CANDIDATE",
    "PARTIAL_PYSPARK",
    "BLOCKED_STUB",
    "GENERATION_FAILED",
    "NO_VERIFIED_PYSPARK_LOGIC",
    "MISSING_FUNCTION"
]:
    count = sum(
        1 for item in readiness_results
        if item["status"] == status
    )

    print(f"{status}: {count}")

print("\nAvailable tables:", len(table_inventory))

print("\nIMPORTANT:")
print("This is static readiness analysis only.")
print("No generated functions were executed.")
print("No test data was inserted.")
print("No target tables were modified.")

# ============================================================
# 5. RETAIN RESULTS FOR THE NEXT NOTEBOOK STEP
# ============================================================

TASKFLOW_READINESS = readiness_results
TASKFLOW_TABLE_INVENTORY = table_inventory

print("\nReadiness results stored in notebook memory.")


In [0]:

import json
import re

# Use the already parsed Taskflow metadata
services = taskflow_input["parser_evidence"]["services"]

DATA_TASK_IDS = {
    "ms07zo8j",
    "ms07zo8z",
    "ms07zo8n",
    "ms07zo8v",
    "ms07zo8r"
}

KEYWORDS = [
    "table", "source", "target", "schema",
    "catalog", "connection", "sql", "query",
    "select", "insert", "update", "merge",
    "path", "file", "mapping", "parameter"
]

def collect_xml_evidence(obj, path="root"):
    results = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            current_path = f"{path}.{key}"

            if isinstance(value, (dict, list)):
                results.extend(
                    collect_xml_evidence(value, current_path)
                )
            elif value is not None:
                text_value = str(value).strip()

                if not text_value:
                    continue

                key_match = any(
                    word in key.lower()
                    for word in KEYWORDS
                )

                value_match = any(
                    word in text_value.lower()
                    for word in KEYWORDS
                )

                if key_match or value_match:
                    results.append({
                        "path": current_path,
                        "value": text_value
                    })

    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            results.extend(
                collect_xml_evidence(
                    value, f"{path}[{index}]"
                )
            )

    return results


print("=" * 85)
print("TASKFLOW SOURCE / TARGET EVIDENCE INSPECTION")
print("=" * 85)

TASKFLOW_XML_EVIDENCE = {}

for service in services:
    service_id = service.get("id")

    if service_id not in DATA_TASK_IDS:
        continue

    service_name = (
        service.get("title")
        or service.get("service_name")
    )

    evidence = collect_xml_evidence(service)

    TASKFLOW_XML_EVIDENCE[service_id] = evidence

    print(f"\n{'=' * 85}")
    print("SERVICE:", service_name)
    print("SERVICE ID:", service_id)
    print("EVIDENCE ITEMS:", len(evidence))

    for item in evidence:
        print(f"\n{item['path']}")
        print("  ", item["value"])

# Also inspect global operations separately.
# These are NOT assigned to individual services
# unless explicit ownership evidence exists.

all_operations = (
    taskflow_input
    .get("parser_evidence", {})
    .get("operations", [])
)

print("\n" + "=" * 85)
print("GLOBAL OPERATION INVENTORY")
print("=" * 85)

print("Total operations:", len(all_operations))

for index, operation in enumerate(all_operations):
    operation_text = json.dumps(
        operation, default=str
    )

    if any(
        keyword in operation_text.lower()
        for keyword in KEYWORDS
    ):
        print(f"\nOperation {index + 1}:")
        print(operation_text[:1500])

print("\nInspection completed. No files or tables modified.")


In [0]:

import json

services = taskflow_input["parser_evidence"]["services"]

DATA_TASK_IDS = {
    "ms07zo8j",
    "ms07zo8z",
    "ms07zo8n",
    "ms07zo8v",
    "ms07zo8r"
}

def inspect_parameter_tree(obj, path="root"):
    results = []

    if isinstance(obj, dict):
        if obj.get("tag") in {"parameter", "operation"}:
            results.append({
                "path": path,
                "tag": obj.get("tag"),
                "name": obj.get("attributes", {}).get("name"),
                "source": obj.get("attributes", {}).get("source"),
                "target": obj.get("attributes", {}).get("to"),
                "value": obj.get("value", obj.get("text"))
            })

        for key, value in obj.items():
            if isinstance(value, (dict, list)):
                results.extend(
                    inspect_parameter_tree(value, f"{path}.{key}")
                )

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            results.extend(
                inspect_parameter_tree(value, f"{path}[{i}]")
            )

    return results


for service in services:
    if service.get("id") not in DATA_TASK_IDS:
        continue

    print("\n" + "=" * 80)
    print("SERVICE:", service.get("title"))
    print("SERVICE ID:", service.get("id"))

    for section in ["service_input", "service_output"]:
        print(f"\n--- {section.upper()} ---")

        evidence = inspect_parameter_tree(
            service.get(section, {}),
            section
        )

        for item in evidence:
            if item["tag"] == "parameter":
                print(
                    "PARAMETER:",
                    item["name"],
                    "| SOURCE:", item["source"],
                    "| VALUE:", item["value"]
                )

            elif item["tag"] == "operation":
                print(
                    "OPERATION:",
                    item["value"],
                    "->",
                    item["target"]
                )

print("\n" + "=" * 80)
print("TASKFLOW GLOBAL INPUT PARAMETERS")
print("=" * 80)

print(json.dumps(
    taskflow_input.get("parser_evidence", {}).get(
        "input_parameters", []
    ),
    indent=2,
    default=str
))


In [0]:
import xml.etree.ElementTree as ET

XML_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "wf_s_m_TIM_STG_UNQ_WRKSTN_NOMTCH_SYNC.TASKFLOW_SANITIZE.xml"
)

tree = ET.parse(XML_PATH)
root = tree.getroot()

patterns = [
    "InputMappingTaskParameterFileDir",
    "InputMappingTaskParameterFileName",
    "SRC_QUERY",
    "SRC_QUERY1",
    "SRC_QUERY2",
    "SRC_QUERY3",
    "LKP_QUERY",
    "TBL_NM",
    "SCHEMA_NAME",
    "parameterFileDir",
    "parameterFileName"
]

print("=== XML PARAMETER / QUERY EVIDENCE ===")

matches = 0

for element in root.iter():
    tag = element.tag.split("}")[-1]
    attrs = dict(element.attrib)
    text_value = (element.text or "").strip()

    combined = " ".join(
        [tag, text_value] +
        [f"{key} {value}" for key, value in attrs.items()]
    )

    if any(pattern.lower() in combined.lower() for pattern in patterns):
        print({
            "tag": tag,
            "attributes": attrs,
            "text": text_value[:500]
        })
        matches += 1

print(f"\nTotal matching XML elements: {matches}")

In [0]:
from pathlib import Path

BASE_PATH = Path("/Volumes/idmc_poc/bronze/idmc_files")

print("=== PARAMETER FILE DISCOVERY ===")

for file_path in sorted(BASE_PATH.rglob("*")):
    if file_path.is_file():
        name = file_path.name.lower()

        if (
            name.endswith(".prm")
            or "parameter" in name
            or "param" in name
        ):
            print(str(file_path))

In [0]:
#1
# ============================================================
# DISPLAY GENERATED PYSPARK CODE FOR ALL MAPPINGS
# ============================================================

GENERATED_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark"
)

MAPPINGS = [
    "m_customer_standardize",
    "m_order_enrichment",
    "m_inventory_reconciliation",
    "m_daily_sales_summary"
]

for mapping_name in MAPPINGS:

    file_path = f"{GENERATED_DIR}/{mapping_name}.py"

    print("\n" + "=" * 100)
    print(f"GENERATED PYSPARK: {mapping_name}")
    print("=" * 100)

    try:
        code = dbutils.fs.head(file_path, 1000000)

        print(code)

        print("\nFile:", file_path)
        print("Status: Successfully read")

    except Exception as e:
        print("Unable to read generated code:", str(e))


In [0]:
import json
import importlib
import inspect

import LLM.Taskflow_Input_builder as taskflow_builder_module

# Reload the latest saved Python file
importlib.invalidate_caches()
importlib.reload(taskflow_builder_module)

TaskflowLLMInputBuilder = (
    taskflow_builder_module.TaskflowLLMInputBuilder
)

print("Loaded module from:")
print(inspect.getfile(TaskflowLLMInputBuilder))

# Rebuild using the updated class
input_builder = TaskflowLLMInputBuilder()

taskflow_input = input_builder.build(
    parsed_taskflow=taskflow_bundle,
    taskflow_graph=taskflow_graph
)

print("\nTOP-LEVEL KEYS:")
print(list(taskflow_input.keys()))

print("\nTASKFLOW INPUT INTEGRITY:")

if "input_integrity" not in taskflow_input:
    raise RuntimeError(
        "The loaded TaskflowLLMInputBuilder does not return "
        "'input_integrity'. Check the module path printed above "
        "and save the updated Taskflow_Input_builder.py file."
    )

print(
    json.dumps(
        taskflow_input["input_integrity"],
        indent=2,
        default=str
    )
)

print("\nTASKFLOW LLM INPUT READY")

In [0]:
import os
from LLM.TaskflowCodeGenerator import TaskflowCodeGenerator

# --------------------------------------------------
# 1. Get Databricks workspace URL
# --------------------------------------------------

workspace_url = (
    spark.conf.get("spark.databricks.workspaceUrl", None)
)

if not workspace_url:
    workspace_url = (
        dbutils.notebook.entry_point
        .getDbutils()
        .notebook()
        .getContext()
        .apiUrl()
        .get()
    )

if not workspace_url.startswith("http"):
    workspace_url = "https://" + workspace_url

# --------------------------------------------------
# 2. Get Databricks access token
# --------------------------------------------------

databricks_token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

if not databricks_token:
    raise RuntimeError("Databricks token is unavailable.")

# --------------------------------------------------
# 3. Validate output directory
# --------------------------------------------------

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "/Workspace/Users/lokeswaraseetaramasastry.k@tcs.com/IDMC_Databricks_LLM/output/taskflow"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --------------------------------------------------
# 4. Initialize generator
# --------------------------------------------------

generator = TaskflowCodeGenerator(
    workspace_url=workspace_url,
    databricks_token=databricks_token,
    model="databricks-gpt-oss-120b",
    max_tokens=16000,
    temperature=0
)

print("Workspace URL:", workspace_url)
print("Token available:", bool(databricks_token))
print("Output directory:", OUTPUT_DIR)

# --------------------------------------------------
# 5. Generate Taskflow migration code
# --------------------------------------------------

generation_result = generator.generate_taskflow_code(
    taskflow_input=taskflow_input,
    output_dir=OUTPUT_DIR,
    max_retries=2
)

print("\nTASKFLOW GENERATION RESULT")
print(generation_result)

In [0]:
from LLM.TaskflowCodeGenerator import TaskflowCodeGenerator
import json

generator = TaskflowCodeGenerator(
    workspace_url=workspace_url,
    databricks_token=databricks_token,
    model="databricks-gpt-oss-120b",
    max_tokens=16000,
    temperature=0
)

generation_result = generator.generate_taskflow_code(
    taskflow_input=taskflow_input,
    output_dir=OUTPUT_DIR,
    max_retries=2
)

print(json.dumps(generation_result, indent=2, default=str))

In [0]:
import os

OUTPUT_DIR = (
    "/Workspace/Users/lokeswaraseetaramasastry.k@tcs.com/"
    "IDMC_Databricks_LLM/output/taskflow"
)

print("Directory exists:", os.path.exists(OUTPUT_DIR))

if os.path.exists(OUTPUT_DIR):
    for file in os.listdir(OUTPUT_DIR):
        print(file)
else:
    print("Taskflow output directory does not exist.")

In [0]:
for file in os.listdir(OUTPUT_DIR):
    file_path = os.path.join(OUTPUT_DIR, file)

    if file.endswith((".py", ".json")):
        print(file_path)

In [0]:
#1
from pathlib import Path
import ast

generated_file = generation_result["generated_file"]

path = Path(generated_file)

if not path.exists():
    raise FileNotFoundError(generated_file)

code = path.read_text(encoding="utf-8")
lines = code.splitlines()

print("=" * 70)
print("GENERATED FILE DIAGNOSTICS")
print("=" * 70)

# 1. Python function definitions
tree = ast.parse(code)

defined_functions = [
    node.name
    for node in ast.walk(tree)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
]

print("\nFUNCTION DEFINITIONS:")
for name in defined_functions:
    print(name)

# 2. Missing task handlers
expected_handlers = (
    generation_result["code_validation"]
    ["task_functions_missing_from_code"]
)

print("\nMISSING HANDLERS:")
for name in expected_handlers:
    print(name)

# 3. Occurrences of run
print("\nOCCURRENCES OF 'run':")

for number, line in enumerate(lines, start=1):
    if "run" in line.lower():
        print(f"{number}: {line}")

# 4. DataFrame operations
print("\nDATAFRAME OPERATIONS:")

for number, line in enumerate(lines, start=1):
    if any(
        term in line
        for term in [
            ".select(",
            ".filter(",
            ".where(",
            ".join(",
            ".groupBy(",
            ".withColumn(",
            ".dropDuplicates(",
            ".write",
            "spark.table("
        ]
    ):
        print(f"{number}: {line}")

print("\nGenerated file:", generated_file)
print("Total lines:", len(lines))
print("Total functions:", len(defined_functions))

In [0]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark"
)

mapping_files = [
    "m_customer_standardize.py",
    "m_order_enrichment.py",
    "m_inventory_reconciliation.py",
    "m_daily_sales_summary.py"
]

for filename in mapping_files:
    file_path = OUTPUT_DIR / filename

    print("\n" + "=" * 100)
    print(f"MAPPING FILE: {filename}")
    print(f"LOCATION: {file_path}")
    print("=" * 100)

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8"))
    else:
        print("File not found:", file_path)